# 📸 snap.py — Web Snapshot Tool (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmatek11/webscraper/blob/main/snap_colab_optimized.ipynb)

Backup stron (HTML + assets + screenshot) albo same screenshoty całych stron, np. **przed i po** zmianach u klienta.

**Jak używać:**
1. Uruchom **1. Instalacja** (raz na sesję, ~1-2 min).
2. Uruchom **2. Pobierz snap.py**, który ściąga aktualną wersję skryptu z GitHuba, więc poprawki w repo działają tu od razu.
3. W **3. Ustawienia i start** wklej adresy, wybierz tryb i uruchom.
4. **4. Wyniki** pobiera ZIP-y na komputer i/lub kopiuje je na Google Drive.

## 1. Instalacja

In [ ]:
# Playwright + Chromium z zależnościami systemowymi
!pip install -q playwright requests
!playwright install --with-deps chromium > /tmp/pw_install.log 2>&1 || tail -30 /tmp/pw_install.log

# Fonty jak na zwykłym komputerze z Windows (Arial, Times, Verdana...) + emoji.
# Bez tego strony bez web-fontów renderują się innym krojem niż u klienta.
!echo "ttf-mscorefonts-installer msttcorefonts/accept-mscorefonts-eula select true" | debconf-set-selections
!apt-get install -y -qq fonts-liberation fonts-noto-color-emoji ttf-mscorefonts-installer > /dev/null 2>&1 || true
!fc-cache -f > /dev/null

In [ ]:
# Test: czy przeglądarka startuje (w osobnym wątku — tak samo jak snap.py)
import threading
def _check():
    from playwright.sync_api import sync_playwright
    try:
        with sync_playwright() as p:
            try:
                b = p.chromium.launch(channel='chromium')
            except Exception:
                b = p.chromium.launch()
            print(f"✅ Instalacja OK — Chromium {b.version} działa")
            b.close()
    except Exception as e:
        print("❌ Przeglądarka nie startuje:\n", e)
t = threading.Thread(target=_check); t.start(); t.join()

## 2. Pobierz snap.py

In [ ]:
BRANCH = "main"  #@param {type:"string"}

import urllib.request
url = f"https://raw.githubusercontent.com/mrmatek11/webscraper/{BRANCH}/snap.py"
urllib.request.urlretrieve(url, "snap.py")
print(f"✅ snap.py pobrany z gałęzi '{BRANCH}'")

## 3. Ustawienia i start

- **URL_LIST**: jeden adres na linię, linie z `#` są pomijane.
- **MODE**: `screenshots` to same PNG (najszybciej), `full` to HTML + assets + screenshot, `crawl` sam znajduje podstrony (sitemap + linki). Prefix `clean-` oznacza agresywne zamykanie popupów.
- **LABEL**: np. `przed` / `po`. Wyniki lądują w osobnych folderach, łatwo porównać.
- **VIEWPORT / SCALE**: `1920x1080` + `1` = typowy monitor Full HD. `1.25` = laptop z Windows (skalowanie 125%). `2` = ostry screen (Retina, plik ~4× większy).

In [ ]:
URL_LIST = """
https://biuro-onesta.pl
"""

MODE = "crawl"  #@param ["screenshots", "full", "crawl", "clean-screenshots", "clean-full", "clean-crawl"]
LABEL = "przed"  #@param {type:"string"}
VIEWPORT = "1920x1080"  #@param ["1920x1080", "1536x864", "1440x900", "1366x768"]
SCALE = 1  #@param [1, 1.25, 1.5, 2] {type:"raw"}
WORKERS = 2  #@param {type:"slider", min:1, max:6, step:1}
# ↑ ile stron naraz. Więcej = szybciej, ale przy dużym crawlu Colab może się wysypać (brak RAM)
MAX_PAGES = 300  #@param {type:"integer"}
MAX_SCREENSHOT_HEIGHT = 30000  #@param {type:"integer"}

import importlib, re
from datetime import datetime
from pathlib import Path
import snap
importlib.reload(snap)

snap.load_config()
vw, vh = map(int, VIEWPORT.split('x'))
snap._CFG.update(
    workers=WORKERS, max_pages=MAX_PAGES, max_screenshot_height=MAX_SCREENSHOT_HEIGHT,
    viewport_width=vw, viewport_height=vh, device_scale_factor=float(SCALE),
)

urls = [l.strip() for l in URL_LIST.splitlines() if l.strip() and not l.strip().startswith('#')]
if MODE.replace('clean-', '', 1) == 'crawl':
    urls = snap.discover_urls(urls, MAX_PAGES)
urls = list(dict.fromkeys(urls))

label = re.sub(r'[^\w\-]', '_', LABEL.strip()) or 'wyniki'
OUTPUT_DIR = Path('/content/results') / f"{datetime.now():%Y-%m-%d}_{label}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"log: {snap.setup_logging(OUTPUT_DIR)}")
print(f"mode: {MODE} | pages: {len(urls)} | viewport: {vw}x{vh} @{SCALE}x | output: {OUTPUT_DIR}")

snap.run(urls, OUTPUT_DIR, mode=MODE)

if not list(OUTPUT_DIR.glob('*.zip')):
    print("\n❌ Nie powstał żaden plik ZIP. Ostatnie linie logu:")
    log = sorted(OUTPUT_DIR.glob('*.log'))
    print(log[-1].read_text()[-3000:] if log else "(brak logu)")
else:
    print("\n✅ Gotowe — uruchom komórkę 4, żeby pobrać wyniki.")


## 4. Wyniki

In [ ]:
DOWNLOAD = True  #@param {type:"boolean"}
COPY_TO_DRIVE = False  #@param {type:"boolean"}
DRIVE_FOLDER = "Web_Backups"  #@param {type:"string"}

import shutil, zipfile
from pathlib import Path

# działa też po restarcie środowiska — bierze najnowszy folder z wynikami
try:
    OUTPUT_DIR
except NameError:
    dirs = sorted(Path('/content/results').glob('*/'), key=lambda d: d.stat().st_mtime)
    OUTPUT_DIR = dirs[-1] if dirs else Path('/content/results')

zips = sorted(OUTPUT_DIR.glob('*.zip'))
if not zips:
    print(f"❌ Brak plików ZIP w {OUTPUT_DIR} — najpierw uruchom komórkę 3 i sprawdź, czy skończyła się bez błędów.")
else:
    for z in zips:
        print(f" - {z.name} ({z.stat().st_size/1024/1024:.1f} MB)")

    # Jeden plik do pobrania — przeglądarki blokują kilka pobrań naraz z Colaba
    if len(zips) == 1:
        bundle = zips[0]
    else:
        bundle = OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}.zip"
        with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_STORED) as zf:
            for f in zips + sorted(OUTPUT_DIR.glob('*.log')):
                zf.write(f, f.name)
    print(f"\n📦 {bundle.name} ({bundle.stat().st_size/1024/1024:.1f} MB)")

    if COPY_TO_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        dest = Path('/content/drive/MyDrive') / DRIVE_FOLDER
        dest.mkdir(parents=True, exist_ok=True)
        shutil.copy2(bundle, dest)
        print(f"✅ skopiowano na Google Drive: {dest / bundle.name}")

    if DOWNLOAD:
        from google.colab import files
        print("⬇️ pobieranie... (jeśli nic się nie pojawi, patrz niżej)")
        files.download(str(bundle))
        print(f"""
Jeśli plik się nie pobrał:
  • kliknij ikonę 📁 (Pliki) po lewej → {' → '.join(bundle.resolve().parts[1:])} → ⋮ → Pobierz
  • albo zaznacz COPY_TO_DRIVE i uruchom tę komórkę jeszcze raz
  • w Safari/Brave: zezwól stronie colab.research.google.com na pobieranie i pliki cookie""")